# Week 7 Leakage-Safe Forecasting Data Preparation

This notebook creates prediction-ready Train, Validation and Test datasets for a 15-minute-ahead Floor 6 AC power forecast. It prepares data only; it does not train or evaluate forecasting models.

## Input and leakage controls

The input is the final main dataset written by `Data_pre/Floor6_Zone1245_Data_Preparation_EDA_executed.ipynb`: `outputs/floor6_zone1245_2018_jul01_sep01_total_ac_eda/floor6_zone1245_cleaned_features_2018_jul01_sep01.csv.gz`. Flagged source imputations are restored to missing before target and feature creation. All lag and rolling features are trailing and zone-specific. Remaining medians are fitted on Train only, and target timestamps are purged at split boundaries.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'ml' and (PROJECT_DIR.parent / 'outputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent
elif PROJECT_DIR.name != 'Artificial Intelligence for Smart-Building Log Analysis':
    candidate = PROJECT_DIR / 'Artificial Intelligence for Smart-Building Log Analysis'
    if candidate.exists():
        PROJECT_DIR = candidate
    else:
        raise FileNotFoundError('Run this notebook from the repository project or ml directory')
sys.path.insert(0, str(PROJECT_DIR / 'ml'))

from forecasting_data_preparation import run_preparation

In [2]:
result = run_preparation()
result['source_quality'], result['split_summary'], result['validation_audit']

({'source_path': 'outputs/floor6_zone1245_2018_jul01_sep01_total_ac_eda/floor6_zone1245_cleaned_features_2018_jul01_sep01.csv.gz',
  'source_file': 'floor6_zone1245_cleaned_features_2018_jul01_sep01.csv.gz',
  'source_notebook': 'Data_pre/Floor6_Zone1245_Data_Preparation_EDA_executed.ipynb',
  'rows': 362880,
  'columns': 38,
  'floor_values': '6',
  'zone_values': '1,2,4,5',
  'zone_count': 4,
  'min_timestamp': '2018-07-01 00:00:00',
  'max_timestamp': '2018-09-01 23:59:00',
  'unique_timestamps': 90720,
  'duplicate_key_rows': 0,
  'non_one_minute_intervals': 0,
  'sampling_frequency': '1 minute',
  'source_sensor_nan_values_before_restoring_flags': 0,
  'originally_missing_total_AC_power_kw': 1267,
  'originally_missing_light_power_kw': 2804,
  'originally_missing_plug_power_kw': 1974,
  'originally_missing_temperature_c': 16540,
  'originally_missing_humidity_pct': 16540,
  'originally_missing_lux': 16540},
 {'train': {'rows': 253380,
   'prediction_min': '2018-07-01 00:00:00',
  

In [3]:
import pandas as pd

reports = PROJECT_DIR / 'outputs/week7_prediction_ready_dataset/reports'
pd.read_csv(reports / 'prediction_ready_manifest.csv')

,split,file,rows,columns,feature_count,prediction_min,prediction_max,target_min,target_max,bytes,sha256
0,train,prediction_ready_train.csv.gz,253380,34,29,2018-07-01 00:00:00,2018-08-13 23:44:00,2018-07-01 00:15:00,2018-08-13 23:59:00,10240584,7b5b913f58333e939075be2587b7f4a62a385502d76791...
1,validation,prediction_ready_validation.csv.gz,56273,34,29,2018-08-14 00:00:00,2018-08-23 23:44:00,2018-08-14 00:15:00,2018-08-23 23:59:00,2405881,68befd9ce98700f34bf2af3ef0faa44ac544118ea6ac8d...
2,test,prediction_ready_test.csv.gz,51780,34,29,2018-08-24 00:00:00,2018-09-01 23:44:00,2018-08-24 00:15:00,2018-09-01 23:59:00,2120938,45cd32b79f090f3a61b362e5ae403d486d534697aeda3f...


## Prediction-ready dataset previews

The following cell reads only the first five rows from each compressed split so that the exported feature and target alignment can be inspected without loading the full datasets into memory.

In [4]:
from IPython.display import display

data_dir = PROJECT_DIR / 'outputs/week7_prediction_ready_dataset/data'

for split in ('train', 'validation', 'test'):
    preview = pd.read_csv(
        data_dir / f'prediction_ready_{split}.csv.gz',
        nrows=5,
    )
    print(f'{split.upper()} - first 5 rows ({preview.shape[1]} columns)')
    with pd.option_context('display.max_columns', None):
        display(preview)

TRAIN - first 5 rows (34 columns)


,timestamp,target_timestamp,floor,zone,total_AC_power_kw,light_power_kw,plug_power_kw,temperature_c,humidity_pct,lux,total_AC_power_kw_was_missing,light_power_kw_was_missing,plug_power_kw_was_missing,temperature_c_was_missing,humidity_pct_was_missing,lux_was_missing,total_AC_power_kw_lag_1,total_AC_power_kw_lag_5,total_AC_power_kw_lag_15,total_AC_power_kw_lag_30,total_AC_power_kw_lag_60,total_AC_power_kw_lag_1440,total_AC_power_kw_rolling_mean_15,total_AC_power_kw_rolling_mean_30,total_AC_power_kw_rolling_mean_60,total_AC_power_kw_rolling_std_15,total_AC_power_kw_rolling_std_60,hour,day_of_week,is_weekend,hour_sin,hour_cos,zone_category,target_total_AC_power_kw_t_plus_15
0,2018-07-01 00:00:00,2018-07-01 00:15:00,6,1,0.0,0.0,0.31,27.57,72.19,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,6,1,0.000000,1.000000,1,0.0
1,2018-07-01 00:01:00,2018-07-01 00:16:00,6,1,0.0,0.0,0.31,27.57,72.18,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,6,1,0.004363,0.999990,1,0.0
2,2018-07-01 00:02:00,2018-07-01 00:17:00,6,1,0.0,0.0,0.31,27.57,72.18,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,6,1,0.008727,0.999962,1,0.0
3,2018-07-01 00:03:00,2018-07-01 00:18:00,6,1,0.0,0.0,0.26,27.56,72.17,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,6,1,0.013090,0.999914,1,0.0
4,2018-07-01 00:04:00,2018-07-01 00:19:00,6,1,0.0,0.0,0.26,27.56,72.17,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,6,1,0.017452,0.999848,1,0.0


VALIDATION - first 5 rows (34 columns)


,timestamp,target_timestamp,floor,zone,total_AC_power_kw,light_power_kw,plug_power_kw,temperature_c,humidity_pct,lux,total_AC_power_kw_was_missing,light_power_kw_was_missing,plug_power_kw_was_missing,temperature_c_was_missing,humidity_pct_was_missing,lux_was_missing,total_AC_power_kw_lag_1,total_AC_power_kw_lag_5,total_AC_power_kw_lag_15,total_AC_power_kw_lag_30,total_AC_power_kw_lag_60,total_AC_power_kw_lag_1440,total_AC_power_kw_rolling_mean_15,total_AC_power_kw_rolling_mean_30,total_AC_power_kw_rolling_mean_60,total_AC_power_kw_rolling_std_15,total_AC_power_kw_rolling_std_60,hour,day_of_week,is_weekend,hour_sin,hour_cos,zone_category,target_total_AC_power_kw_t_plus_15
0,2018-08-14 00:00:00,2018-08-14 00:15:00,6,1,0.0,0.0,0.22,29.29,69.89,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0.000000,1.000000,1,0.0
1,2018-08-14 00:01:00,2018-08-14 00:16:00,6,1,0.0,0.0,0.26,29.29,69.89,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0.004363,0.999990,1,0.0
2,2018-08-14 00:02:00,2018-08-14 00:17:00,6,1,0.0,0.0,0.26,29.29,69.89,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0.008727,0.999962,1,0.0
3,2018-08-14 00:03:00,2018-08-14 00:18:00,6,1,0.0,0.0,0.26,29.29,69.89,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0.013090,0.999914,1,0.0
4,2018-08-14 00:04:00,2018-08-14 00:19:00,6,1,0.0,0.0,0.26,29.29,69.88,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0.017452,0.999848,1,0.0


TEST - first 5 rows (34 columns)


,timestamp,target_timestamp,floor,zone,total_AC_power_kw,light_power_kw,plug_power_kw,temperature_c,humidity_pct,lux,total_AC_power_kw_was_missing,light_power_kw_was_missing,plug_power_kw_was_missing,temperature_c_was_missing,humidity_pct_was_missing,lux_was_missing,total_AC_power_kw_lag_1,total_AC_power_kw_lag_5,total_AC_power_kw_lag_15,total_AC_power_kw_lag_30,total_AC_power_kw_lag_60,total_AC_power_kw_lag_1440,total_AC_power_kw_rolling_mean_15,total_AC_power_kw_rolling_mean_30,total_AC_power_kw_rolling_mean_60,total_AC_power_kw_rolling_std_15,total_AC_power_kw_rolling_std_60,hour,day_of_week,is_weekend,hour_sin,hour_cos,zone_category,target_total_AC_power_kw_t_plus_15
0,2018-08-24 00:00:00,2018-08-24 00:15:00,6,1,0.0,0.0,0.28,27.14,62.41,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,4,0,0.000000,1.000000,1,0.0
1,2018-08-24 00:01:00,2018-08-24 00:16:00,6,1,0.0,0.0,0.29,27.15,62.40,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,4,0,0.004363,0.999990,1,0.0
2,2018-08-24 00:02:00,2018-08-24 00:17:00,6,1,0.0,0.0,0.28,27.15,62.39,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,4,0,0.008727,0.999962,1,0.0
3,2018-08-24 00:03:00,2018-08-24 00:18:00,6,1,0.0,0.0,0.24,27.16,62.38,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,4,0,0.013090,0.999914,1,0.0
4,2018-08-24 00:04:00,2018-08-24 00:19:00,6,1,0.0,0.0,0.28,27.16,62.37,0.0,False,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,4,0,0.017452,0.999848,1,0.0


Generated datasets are stored under `outputs/week7_prediction_ready_dataset/data/` and are intentionally excluded from Git because they are large and reproducible. The lightweight manifest, training medians, configuration and leakage audit are stored under `outputs/week7_prediction_ready_dataset/reports/`.